In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "anndata>=0.12.11",
#     "scanpy>=1.12.1",
#     "spatialdata>=0.7.2",
#     "zarr>=3.1.6",
# ]
#
# [tool.uv]
# exclude-newer = "2026-04-30T12:53:14.229210378+02:00"
# ///

In [ ]:
import shutil

import anndata as ad
import numpy as np
import pandas as pd
import zarr
from scipy.sparse import csc_matrix

In [ ]:
export_zarr_v3 = False

if export_zarr_v3:
    ad.settings.zarr_write_format = 3
    ad.settings.write_csr_csc_indices_with_min_possible_dtype = True
    ad.settings.auto_shard_zarr_v3 = True

In [ ]:
gene_map = {
    "ACTB": "ENSG00000075624",  # Housekeeping
    "CD3E": "ENSG00000198851",  # T-cell marker
    "CD79A": "ENSG00000105369",  # B-cell marker
    "GAPDH": "ENSG00000111640",  # Housekeeping
    "GNLY": "ENSG00000115523",  # NK-cell marker
    "LYZ": "ENSG00000090382",  # Monocyte marker
}

gene_symbols = list(gene_map.keys())
ensembl_ids = list(gene_map.values())

(gene_symbols, ensembl_ids)

In [ ]:
n_cells = 100
n_genes = len(gene_symbols)

In [ ]:
# 2. Create the Observation Metadata (Cells)
obs = pd.DataFrame(index=[f"Cell_{i:03d}" for i in range(n_cells)])
obs["cell_type"] = np.random.choice(["T-cell", "B-cell", "Monocyte"], size=n_cells)
obs["treatment"] = np.random.choice(["treated", "untreated"], size=n_cells)

In [ ]:
# 3. Create the Variable Metadata (Genes)
var = pd.DataFrame(index=gene_symbols)
var["symbol"] = gene_symbols
var["ensembl_gene_id"] = ensembl_ids
var["feature_type"] = "Random Counts"

In [ ]:
counts_dense = np.random.poisson(lam=2.0, size=(n_cells, len(gene_symbols))).astype(np.float32)
counts_csc = csc_matrix(counts_dense)

In [ ]:
adata = ad.AnnData(X=counts_csc, obs=obs, var=var)

In [ ]:
adata.obsm["X_umap"] = np.random.normal(size=(n_cells, 2))
adata.obsm["tsne"] = np.random.normal(size=(n_cells, 2))

In [ ]:
# 4. Final Metadata Preparation
# App requires _index attribute for the unique identifiers
adata.obs.index.name = "cell_id"
adata.var.index.name = "gene_symbol"

adata.obs.index = adata.obs.index.astype(object)
adata.var.index = adata.var.index.astype(object)

# 2. Fix the Columns (the metadata)
for df in [adata.obs, adata.var]:
    for col in df.columns:
        # If it's a Categorical or an Arrow-backed string or str, flatten it
        if pd.api.types.is_categorical_dtype(df[col]) or "Arrow" in str(df[col].dtype) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype(object)
        else:
            print(f"Column '{col}' is already a standard type: {df[col].dtype}")

In [ ]:
folder = "test-data"
datasetname = f"single-cell-{"v3" if export_zarr_v3 else "v2"}-test-data"
path = f"{folder}/{datasetname}.zarr"

adata.write_zarr(path, chunks=(n_cells, len(gene_symbols)))


zarr.consolidate_metadata(path)

# zip the zarr directory for easier sharing
# shutil.make_archive(datasetname, "zip", f"{datasetname}.zarr")

# delete the original zarr directory to save space
# shutil.rmtree(f"{datasetname}.zarr")